<a href="https://colab.research.google.com/github/JeanMusenga/SZUniversity/blob/main/Ablating_DML_CPArchISLinker(RQ5_DML).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# RQ5-DML: CPArchISLinker without DML
# ==========================================================
!pip install -q sentence-transformers rapidfuzz openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.7 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random, re
from sentence_transformers import SentenceTransformer
from rapidfuzz import fuzz
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
import matplotlib.pyplot as plt

# ------------------ Reproducibility ------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
# ==========================================================
# Load Dataset
# ==========================================================
PATH = "Dataset_GitHub_Solutions.xlsx"
df = pd.read_excel(PATH)
required = {'commit_id','so_id','p_text','s_text','label'}
assert required.issubset(df.columns)
df['label'] = df['label'].astype(int)

# ------------------ Commit-level 80/20 split ------------------
commit_ids = df.commit_id.unique()
train_cids, test_cids = train_test_split(commit_ids, test_size=0.2, random_state=SEED)

df_train = df[df.commit_id.isin(train_cids)].copy()
df_test  = df[df.commit_id.isin(test_cids)].copy()

print("Train commits:", len(train_cids))
print("Test commits :", len(test_cids))

Train commits: 786
Test commits : 197


# I. Preprocessing Layer

In [ ]:
# ==========================================================
# I. Preprocessing
# ==========================================================
def clean_text(s):
    if pd.isna(s): return ""
    s = str(s).lower()
    s = re.sub(r"`{3}.*?`{3}", " ", s, flags=re.S)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"&[a-zA-Z]+;", " ", s)
    s = re.sub(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+",
        '',
        s,
        flags=re.UNICODE
    )
    s = re.sub(r"\s+", " ", s)
    return s.strip()

for d in [df_train, df_test]:
    d["p_text"] = d["p_text"].apply(clean_text)
    d["s_text"] = d["s_text"].apply(clean_text)

# II. Feature Extraction

In [ ]:
# ==========================================================
# II. Feature Extraction
# ==========================================================

sbert = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# ---------- Embedding Cache ----------
_EMB = {}

def embed(text: str) -> np.ndarray:
    text = text.strip().lower()
    if text not in _EMB:
        _EMB[text] = sbert.encode(
            text,
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype(np.float32)
    return _EMB[text]

# ---------- Precompute Commit Keywords ----------
commit_kw = {
    cid: re.findall(r"\b\w+\b", txt)
    for cid, txt in zip(df.commit_id, df.p_text)
}

# ---------- Lexical Feature (L) ----------
def lexical_score(commit_id, s_text):
    kws = commit_kw.get(commit_id, [])
    if not kws:
        return 0.0
    return np.mean([fuzz.partial_ratio(k, s_text) / 100 for k in kws])

# ---------- Semantic Similarity (S) ----------
def semantic_score(vp, vs):
    return cosine_similarity(vp.reshape(1, -1), vs.reshape(1, -1))[0, 0]

# ---------- Association Rules (A) ----------
PROBLEM_TERMS = {
    "performance": ["latency", "slow", "bottleneck", "throughput", "performance", "latency"],
    "scalability": ["scale", "scaling", "load", "traffic", "scalability",  "high traffic"],
    "security": ["authentication", "authorization", "vulnerability", "security", "authentication", "encryption", "vulnerability"],
    "maintainability": ["refactor", "technical debt", "maintainability",  "clean code"],
    "reliability": ["crash", "failure", "availability", "reliability", "fault"],
    "deployment": ["deployment", "docker", "kubernetes", "ci/cd"],
    "communication": ["communication", "rpc", "rest", "http", "message"],
    "anti_pattern": ["god class", "spaghetti", "anti-pattern"],
}

SOLUTION_TERMS = {
    "architectural_pattern": ["microservice", "mvc", "layered", "event-driven"],
    "architectural_tactic": ["caching", "load balancing", "replication"],
    "framework": ["spring", "django", "react", "flask"],
    "api": ["api", "rest", "grpc"],
    "protocol": ["http", "https", "tcp"],
    "refactoring": ["refactor", "restructure", "redesign"],
}

ACTION_VERBS = {
    "optimize", "refactor", "migrate", "replace",
    "remove", "add", "improve", "reduce", "fix"
}

def contains_any(text, terms):
    return any(t in text for t in terms)

def extract_actions(text):
    return {v for v in ACTION_VERBS if v in text}

def association_score(p_text, s_text):
    p = p_text.lower()
    s = s_text.lower()

    rule_1 = any(
        contains_any(p, v) and contains_any(s, v)
        for v in PROBLEM_TERMS.values()
    )

    rule_2 = any(
        contains_any(p, v) and contains_any(s, v)
        for v in SOLUTION_TERMS.values()
    )

    rule_3 = bool(extract_actions(p) & extract_actions(s))

    return np.mean([rule_1, rule_2, rule_3])

# ---------- MCI ----------
def compute_mci(L, S, A, α=0.3, β=0.4, γ=0.3):
    return α * L + β * S + γ * A

# ---------- Feature Vector ----------
def build_features(row):
    vp = embed(row.p_text)
    vs = embed(row.s_text)

    L = lexical_score(row.commit_id, row.s_text)
    S = semantic_score(vp, vs)
    A = association_score(row.p_text, row.s_text)
    MCI = compute_mci(L, S, A)

    # Fixed-size numeric feature vector
    return np.array([L, S, A, MCI], dtype=np.float32)

# ---------- Matrix Builder ----------
def build_matrix(df):
    X = np.vstack(df.apply(build_features, axis=1))
    y = df.label.values.astype(np.float32)
    return X, y


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Train/Test Split

# III Relevancy Identification Layer

## Stage-1 Training -

## Stage-1 Evaluation

# IV. Linking and Recommendation Layer

In [ ]:
# ==========================================================
# CPArchISLinker–w/o–DML (Stage 1 removed)
# ==========================================================

# II. Feature Extraction (already defined above)
# Keep Lexical (L), Semantic (S), Association (A), and MCI
# Stage-1 probabilities are removed

X_train_raw, y_train = build_matrix(df_train)
X_test_raw,  y_test  = build_matrix(df_test)

scaler = StandardScaler().fit(X_train_raw)

X_train = scaler.transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

X_train_t = torch.tensor(X_train, device=device)
X_test_t  = torch.tensor(X_test, device=device)

y_train_t = torch.tensor(y_train, device=device)
y_test_t  = torch.tensor(y_test, device=device)

# ==========================================================
# Stage-2: Linking & Recommendation without DML
# ==========================================================

# We no longer have Stage-1 probabilities
# For cross-commit similarity, we use raw Stage-2 features as proxy embeddings
# ==========================================================
# Build Stage-2 training pairs for CPArchISLinker–w/o–DML
# ==========================================================
def build_stage2_pairs(df_split, candidates, all_feats):
    """
    Generate positive-negative pairs for pairwise ranking.
    Positive: relevant pairs (label==1)
    Negative: candidate pairs from cross-commit retrieval
    """
    pairs = []

    for cid, pos_list in candidates.items():
        g = df_split[df_split.commit_id == cid]

        # Positive features (label==1)
        pos_feats = [
            all_feats[df_split.index.get_loc(i)]
            for i in g[g.label == 1].index
        ]

        # Negative features: any candidate not in positive set
        neg_feats = [
            all_feats[p]
            for p in pos_list
            if not any(np.array_equal(all_feats[p], pf) for pf in pos_feats)
        ]

        # Build all positive-negative pairs
        for fp in pos_feats:
            for fn in neg_feats:
                pairs.append((fp, fn))

    return pairs

def build_stage2_candidates_ablate(df_split, X_raw, top_m=30):
    """
    Build Stage-2 candidate indices and features for CPArchISLinker without DML.
    Uses raw Stage-2 features only (Lexical + Semantic + Association + MCI).
    """
    all_feats = X_raw.copy()  # use raw Stage-2 features
    candidates = {}

    for cid, g in df_split.groupby("commit_id"):
        # Own commit positions
        own_pos = [df_split.index.get_loc(i) for i in g.index]
        anchor_idx = own_pos[0]

        # Compute cross-commit similarity using Stage-2 raw features
        sims = cosine_similarity(
            all_feats[anchor_idx].reshape(1, -1),
            all_feats
        ).flatten()

        tmp = df_split.copy()
        tmp["sim"] = sims
        tmp["pos"] = range(len(tmp))

        # Exclude same commit
        tmp = tmp[tmp.commit_id != cid]

        # Take top-m cross-commit candidates
        cross_pos = tmp.sort_values("sim", ascending=False).head(top_m)["pos"].tolist()

        # Merge own + cross-commit positions
        merged_pos = list(set(own_pos + cross_pos))
        candidates[cid] = merged_pos

    return candidates, all_feats

# ---------------- Build Stage-2 candidates ----------------
train_candidates, train_all_feats = build_stage2_candidates_ablate(
    df_train, X_train
)

train_pairs = build_stage2_pairs(
    df_train, train_candidates, train_all_feats
)

print("Stage-2 training pairs (w/o DML):", len(train_pairs))


# ==========================================================
# RankNet model
# ==========================================================
class RankNet(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze()

ranker = RankNet(train_pairs[0][0].shape[0]).to(device)
opt2 = torch.optim.Adam(ranker.parameters(), lr=1e-3)

def pairwise_rank_loss(sp, sn, margin=1.0):
    return torch.mean(torch.clamp(margin - (sp - sn), min=0))

# RankNet model remains unchanged
ranker = RankNet(train_pairs[0][0].shape[0]).to(device)
opt2 = torch.optim.Adam(ranker.parameters(), lr=1e-3)


Stage-2 training pairs (w/o DTML): 25244


## Stage-2 Training

In [ ]:
# Stage-2 Training
EPOCHS_STAGE2 = 40

for epoch in range(EPOCHS_STAGE2):
    random.shuffle(train_pairs)
    total = 0.0

    for fp, fn in train_pairs:
        fp = torch.tensor(fp, device=device)
        fn = torch.tensor(fn, device=device)

        loss = pairwise_rank_loss(ranker(fp), ranker(fn))
        opt2.zero_grad()
        loss.backward()
        opt2.step()

        total += loss.item()

    print(f"[Stage-2 w/o DML] Epoch {epoch+1:02d} | Loss {total/len(train_pairs):.4f}")



[Stage-2 w/o DTML] Epoch 01 | Loss 0.8656
[Stage-2 w/o DTML] Epoch 02 | Loss 0.8391
[Stage-2 w/o DTML] Epoch 03 | Loss 0.8327
[Stage-2 w/o DTML] Epoch 04 | Loss 0.8278
[Stage-2 w/o DTML] Epoch 05 | Loss 0.8254
[Stage-2 w/o DTML] Epoch 06 | Loss 0.8230
[Stage-2 w/o DTML] Epoch 07 | Loss 0.8218
[Stage-2 w/o DTML] Epoch 08 | Loss 0.8195
[Stage-2 w/o DTML] Epoch 09 | Loss 0.8177
[Stage-2 w/o DTML] Epoch 10 | Loss 0.8201
[Stage-2 w/o DTML] Epoch 11 | Loss 0.8201
[Stage-2 w/o DTML] Epoch 12 | Loss 0.8174
[Stage-2 w/o DTML] Epoch 13 | Loss 0.8141
[Stage-2 w/o DTML] Epoch 14 | Loss 0.8161
[Stage-2 w/o DTML] Epoch 15 | Loss 0.8138
[Stage-2 w/o DTML] Epoch 16 | Loss 0.8102
[Stage-2 w/o DTML] Epoch 17 | Loss 0.8128
[Stage-2 w/o DTML] Epoch 18 | Loss 0.8107
[Stage-2 w/o DTML] Epoch 19 | Loss 0.8090
[Stage-2 w/o DTML] Epoch 20 | Loss 0.8087
[Stage-2 w/o DTML] Epoch 21 | Loss 0.8087
[Stage-2 w/o DTML] Epoch 22 | Loss 0.8074
[Stage-2 w/o DTML] Epoch 23 | Loss 0.8075
[Stage-2 w/o DTML] Epoch 24 | Loss

## Stage-2 Evaluation

In [ ]:
# ==========================================================
# Stage-2 Evaluation (w/o DML)
# ==========================================================
def evaluate_stage2_ablate(df_split, X_raw, ranker, top_m=30, ks=[1,3,5,10]):
    """
    Evaluate Stage-2 cross-commit retrieval without DML.
    Uses raw Stage-2 features for cross-commit similarity.
    """
    # Build candidates using ablated candidate builder
    candidates, all_feats = build_stage2_candidates_ablate(df_split, X_raw, top_m=top_m)

    hr = {k: 0 for k in ks}
    mrr = []

    for cid, pos_list in candidates.items():
        scores, labels = [], []

        for p in pos_list:
            with torch.no_grad():
                s = ranker(torch.tensor(all_feats[p], device=device)).item()
            scores.append(s)
            labels.append(df_split.iloc[p].label)

        order = np.argsort(scores)[::-1]
        labels = np.array(labels)[order]

        pos = np.where(labels == 1)[0]
        mrr.append(1 / (pos[0] + 1) if len(pos) > 0 else 0)

        for k in ks:
            hr[k] += int(labels[:k].sum() > 0)

    n = df_split.commit_id.nunique()
    hr = {f"HR@{k}": hr[k]/n for k in ks}
    return hr, np.mean(mrr)

# ---------------- Run evaluation ----------------
hr, mrr = evaluate_stage2_ablate(df_test, X_test_raw, ranker)

print("\nStage-2 Cross-Commit Results (w/o DML)")
print("HR:", hr)
print("MRR:", mrr)



Stage-2 Cross-Commit Results (w/o DTML)
HR: {'HR@1': 0.01015228426395939, 'HR@3': 0.8883248730964467, 'HR@5': 0.9289340101522843, 'HR@10': 0.9746192893401016}
MRR: 0.4639212566309815


In [ ]:
# ==========================================================
# Save Cross-Commit Top-K Results (w/o DML)
# ==========================================================
top_k = 10

# Use the ablated candidate builder
test_candidates, test_all_feats = build_stage2_candidates_ablate(
    df_test, X_test_raw, top_m=30
)

rows = []

for cid, pos_list in test_candidates.items():
    scores = []

    for p in pos_list:
        with torch.no_grad():
            s = ranker(
                torch.tensor(test_all_feats[p], device=device)
            ).item()
        scores.append(s)

    order = np.argsort(scores)[::-1][:top_k]

    for r, idx in enumerate(order, 1):
        row = df_test.iloc[pos_list[idx]].copy()
        row["stage2_score"] = scores[idx]
        row["stage2_rank"] = r
        rows.append(row)

df_stage2_topk = pd.DataFrame(rows)
df_stage2_topk.to_excel(
    "DML_Ablated_Stage2_CrossCommit_TopK_Ranked_Results.xlsx",
    index=False
)

print("\nSaved: DML_Ablated_Stage2_CrossCommit_TopK_Ranked_Results.xlsx")



Saved: DTML_Ablated_Stage2_CrossCommit_TopK_Ranked_Results.xlsx
